In [7]:
import numpy as np
import pandas as pd

def recall_at_k(actual, predicted, k=5):
    """
    actual: список ID действительно кликнутых новостей [item_id]
    predicted: отранжированный список рекомендаций модели [item1, item2, item3, ...]
    """
    predicted_at_k = predicted[:k]
    hits = len(set(actual) & set(predicted_at_k))
    return hits / len(actual) if len(actual) > 0 else 0.0

def ndcg_at_k(actual, predicted, k=5):
    """
    Расчет NDCG@k для одной сессии
    """
    predicted_at_k = predicted[:k]
    dcg = 0.0
    for i, p in enumerate(predicted_at_k):
        if p in actual:
            dcg += 1.0 / np.log2(i + 2) # i+2 т.к. индексация с 0

    # Идеальный DCG (IDCG)
    idcg = sum([1.0 / np.log2(i + 2) for i in range(min(len(actual), k))])

    return dcg / idcg if idcg > 0 else 0.0

# Пример проверки метрик на тесте:
actual_clicks = ['N12345']
model_recommendations = ['N99999', 'N12345', 'N55555', 'N77777', 'N88888']

print(f"Recall@5: {recall_at_k(actual_clicks, model_recommendations, k=5):.4f}")
print(f"NDCG@5: {ndcg_at_k(actual_clicks, model_recommendations, k=5):.4f}")

Recall@5: 1.0000
NDCG@5: 0.6309


In [8]:
behav_cols = ['impression_id', 'user_id', 'time', 'history', 'impressions']
behav_df = pd.read_csv('../data/behaviors.tsv', sep='\t', names=behav_cols)
behav_df['datetime'] = pd.to_datetime(behav_df['time'], format='%m/%d/%Y %I:%M:%S %p')

In [9]:
# --- ИСПРАВЛЕНИЕ ОШИБКИ "time" ---
print(f"Строк ДО очистки: {len(behav_df)}")
behav_df = behav_df[behav_df['time'] != 'time'].copy()
print(f"Строк ПОСЛЕ очистки: {len(behav_df)}")
# ---------------------------------

Строк ДО очистки: 156965
Строк ПОСЛЕ очистки: 156965


In [10]:

train_df=behav_df[behav_df['datetime']<'14-11-2019']
test_df=behav_df[behav_df['datetime']>='14-11-2019']
print(len(train_df))
print(len(test_df))

126695
30270


In [11]:
display(train_df.head())
display(test_df.head())

,impression_id,user_id,time,history,impressions,datetime
0,1,U13740,11/11/2019 9:05:58 AM,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N55689-1 N35729-0,2019-11-11 09:05:58
1,2,U91836,11/12/2019 6:11:30 PM,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...,2019-11-12 18:11:30
3,4,U34670,11/11/2019 5:28:05 AM,N45729 N2203 N871 N53880 N41375 N43142 N33013 ...,N35729-0 N33632-0 N49685-1 N27581-0,2019-11-11 05:28:05
4,5,U8125,11/12/2019 4:11:21 PM,N10078 N56514 N14904 N33740,N39985-0 N36050-0 N16096-0 N8400-1 N22407-0 N6...,2019-11-12 16:11:21
5,6,U19739,11/11/2019 6:52:13 PM,N39074 N14343 N32607 N32320 N22007 N442 N19001...,N21119-1 N53696-0 N33619-1 N25722-0 N2869-0,2019-11-11 18:52:13


,impression_id,user_id,time,history,impressions,datetime
2,3,U73700,11/14/2019 7:01:48 AM,N10732 N25792 N7563 N21087 N41087 N5445 N60384...,N50014-0 N23877-0 N35389-0 N49712-0 N16844-0 N...,2019-11-14 07:01:48
10,11,U89744,11/14/2019 8:38:04 AM,N24422 N25287 N39121 N41777 N58226 N119 N29197...,N47572-0 N45523-0 N64560-0 N53245-0 N8509-0 N5...,2019-11-14 08:38:04
13,14,U29155,11/14/2019 12:26:47 PM,N60785 N11885 N38939 N25114 N44984 N4830 N2068...,N44698-0 N37204-0 N36612-0 N64174-0 N29212-0 N...,2019-11-14 12:26:47
20,21,U70879,11/14/2019 10:45:51 AM,N47823 N44013 N17354 N26531 N22570 N16215 N298...,N38442-0 N50601-0 N36016-0 N42457-0 N23446-0 N...,2019-11-14 10:45:51
39,40,U27024,11/14/2019 2:24:04 PM,N38629 N50155 N29177 N56426 N63842 N36565 N307...,N20394-0 N28072-0 N29212-0 N47572-0 N54321-0 N...,2019-11-14 14:24:04


In [12]:
train_df.to_csv('../data/train_df.tsv', sep='\t')
test_df.to_csv('../data/test_df.tsv', sep='\t')

In [13]:
from collections import Counter

# 1. Считаем популярность новостей НА TRAIN (чтобы не было утечки из будущего)
train_clicks = []
for imp in train_df['impressions'].dropna():
    for item in imp.split():
        if item.endswith('-1'):
            train_clicks.append(item.split('-')[0])

pop_counter = Counter(train_clicks)

# 2. Метрики (NDCG@5 и Recall@5)
def recall_at_k(actual, predicted, k=5):
    predicted_at_k = predicted[:k]
    hits = len(set(actual) & set(predicted_at_k))
    return hits / len(actual) if len(actual) > 0 else 0.0

def ndcg_at_k(actual, predicted, k=5):
    predicted_at_k = predicted[:k]
    dcg = 0.0
    for i, p in enumerate(predicted_at_k):
        if p in actual:
            dcg += 1.0 / np.log2(i + 2)
    idcg = sum([1.0 / np.log2(i + 2) for i in range(min(len(actual), k))])
    return dcg / idcg if idcg > 0 else 0.0

# 3. Оценка бейзлайнов на TEST
random_ndcg, random_recall = [], []
pop_ndcg, pop_recall = [], []

for idx, row in test_df.iterrows():
    imp_str = row['impressions']
    if pd.isna(imp_str):
        continue

    items = imp_str.split()
    candidates = [x.split('-')[0] for x in items]
    actual_clicks = [x.split('-')[0] for x in items if x.endswith('-1')]

    if not actual_clicks:
        continue

    # Baseline 1: Random
    np.random.seed(42 + idx % 1000)
    rand_preds = list(candidates)
    np.random.shuffle(rand_preds)

    random_ndcg.append(ndcg_at_k(actual_clicks, rand_preds, k=5))
    random_recall.append(recall_at_k(actual_clicks, rand_preds, k=5))

    # Baseline 2: Popularity (ранжируем кандидатов по кликам из train)
    pop_preds = sorted(candidates, key=lambda x: pop_counter.get(x, 0), reverse=True)

    pop_ndcg.append(ndcg_at_k(actual_clicks, pop_preds, k=5))
    pop_recall.append(recall_at_k(actual_clicks, pop_preds, k=5))

print("=== РЕЗУЛЬТАТЫ БЕЙЗЛАЙНОВ (на Test set) ===")
print(f"1. Random Baseline:    NDCG@5 = {np.mean(random_ndcg):.4f} | Recall@5 = {np.mean(random_recall):.4f}")
print(f"2. Popularity Baseline: NDCG@5 = {np.mean(pop_ndcg):.4f} | Recall@5 = {np.mean(pop_recall):.4f}")

=== РЕЗУЛЬТАТЫ БЕЙЗЛАЙНОВ (на Test set) ===
1. Random Baseline:    NDCG@5 = 0.2028 | Recall@5 = 0.3078
2. Popularity Baseline: NDCG@5 = 0.2168 | Recall@5 = 0.3253
